# PDF RAG Chatbot

PDF Loader → Chunks → Embeddings → FAISS Vector Store → Retriever → Prompt → LLM → Answer

In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-groq faiss-cpu pypdf sentence-transformers


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import os

C:\Users\ASUS\AppData\Local\Temp\ipykernel_10164\353806670.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 1. Load the PDF

Put your institute PDF in the same folder as this notebook and update the filename below.

In [3]:
pdf_path = "govardhan_institute_demo_knowledge_base.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Total pages:", len(documents))
print("\n--- First page ---\n")
print(documents[0].page_content[:3000])

Total pages: 3

--- First page ---

Demo RAG Knowledge Base  Page 1
 Govardhan Institute & Infotech
 Demo Knowledge Base for RAG Chatbot Training
Important: This document is a fictional/demo knowledge base created for teaching RAG. It is not an official
institutional brochure and should not be treated as verified business information.
1. About the Institute
Govardhan Institute & Infotech is presented in this demo as a technology education and training institute
based in Surat, Gujarat. The institute focuses on practical, career-oriented technology education and
hands-on learning.
The demo organization provides training in programming, web development, application development,
databases, data analysis, artificial intelligence, generative AI and related software technologies.
2. Learning Approach
The institute follows a practical learning model. Students are introduced to concepts, practice them through
exercises, build small applications and then work on larger projects. The goal is to

## 2. Split the PDF into chunks

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))
print("\n--- Example chunk ---\n")
print(chunks[0].page_content)

Total chunks: 7

--- Example chunk ---

Demo RAG Knowledge Base  Page 1
 Govardhan Institute & Infotech
 Demo Knowledge Base for RAG Chatbot Training
Important: This document is a fictional/demo knowledge base created for teaching RAG. It is not an official
institutional brochure and should not be treated as verified business information.
1. About the Institute
Govardhan Institute & Infotech is presented in this demo as a technology education and training institute
based in Surat, Gujarat. The institute focuses on practical, career-oriented technology education and
hands-on learning.
The demo organization provides training in programming, web development, application development,
databases, data analysis, artificial intelligence, generative AI and related software technologies.
2. Learning Approach
The institute follows a practical learning model. Students are introduced to concepts, practice them through
exercises, build small applications and then work on larger projects. The goal i

## 3. Create embeddings

In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_embedding = embeddings.embed_query("What courses are available?")
print("Embedding vector size:", len(test_embedding))
print("First 10 values:", test_embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding vector size: 384
First 10 values: [0.042240630835294724, -0.08092272281646729, 0.02484465017914772, 0.045449066907167435, -0.08849178999662399, -0.014352134428918362, -0.07448142021894455, -0.04312124475836754, -0.02418208308517933, 0.08113031089305878]


## 4. Create the FAISS vector store

In [6]:
vector_store = FAISS.from_documents(chunks, embeddings)
print("Vector store created successfully.")

Vector store created successfully.


## 5. Create the retriever

In [7]:
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
print("Retriever is ready.")

Retriever is ready.


## 6. Test retrieval before using the LLM

The retriever only finds relevant PDF content. It does not generate the answer.

In [19]:
query = "What courses and Technologies are available at Govardhan Institute?"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Retrieved Document {i} ---")
    print(doc.page_content)
    print("Page:", doc.metadata.get("page", "unknown"))


--- Retrieved Document 1 ---
applications, REST APIs and AI-powered chat applications.
8. Certificates
For this demo knowledge base, assume that eligible students can receive a course or program completion
certificate after satisfying the applicable course requirements. Exact certificate rules should always be
confirmed with the administration.
9. Batch Information
Batch timing, course duration, fees, availability and start dates can vary by program and batch. The chatbot
should not invent these details when they are not present in the retrieved knowledge base.
10. Admission and Enrollment
Students interested in joining a program may ask about the course syllabus, duration, fees, schedule,
prerequisites, projects and certificate. If the knowledge base does not contain the requested information, the
assistant should direct the student to the administration.
11. Contact Information
For questions that cannot be answered from this knowledge base, contact the Govardhan Institute & Infotech

## 7. Configure the LLM

Set your Groq API key before running this cell.

In [ ]:
# os.environ["GROQ_API_KEY"] = "your_api_key"

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2
)

print("LLM ready.")

LLM ready.


## 8. Format retrieved documents into context

In [10]:
def format_docs(docs):
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', 0) + 1}]\n{doc.page_content}"
        for doc in docs
    )

context = format_docs(retrieved_docs)
print(context)

[Page 2]
applications, REST APIs and AI-powered chat applications.
8. Certificates
For this demo knowledge base, assume that eligible students can receive a course or program completion
certificate after satisfying the applicable course requirements. Exact certificate rules should always be
confirmed with the administration.
9. Batch Information
Batch timing, course duration, fees, availability and start dates can vary by program and batch. The chatbot
should not invent these details when they are not present in the retrieved knowledge base.
10. Admission and Enrollment
Students interested in joining a program may ask about the course syllabus, duration, fees, schedule,
prerequisites, projects and certificate. If the knowledge base does not contain the requested information, the
assistant should direct the student to the administration.
11. Contact Information
For questions that cannot be answered from this knowledge base, contact the Govardhan Institute & Infotech

[Page 1]
Demo RAG K

## 9. Design the RAG prompt

The LLM must use only the retrieved PDF context. If the context is insufficient, it gives the administration contact.

In [11]:
ADMIN_PHONE = "9898 57 68 77"

prompt = ChatPromptTemplate.from_template(f'''
You are the official AI assistant for Govardhan Institute and Infotech.

Answer the user's question using ONLY the information in the PDF context.

Context:
{{context}}

User Question:
{{question}}

Rules:
1. Use only the provided context.
2. Do not invent or assume information.
3. Answer clearly when the context supports the answer.
4. If the context does not contain enough information, do not guess.
5. In that case, say:
"I don't have enough information in the provided institute information
to answer this question. Please contact the Govardhan Institute and
Infotech administration at {ADMIN_PHONE}."
6. Do not claim that you contacted an administrator.
7. When useful, mention the PDF page number.

Answer:
''')

## 10. Inspect the exact prompt

This is useful for teaching students what the LLM actually receives.

In [12]:
question = "What courses and Technologies are available at Govardhan Institute?"
docs = retriever.invoke(question)
context = format_docs(docs)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

print(final_prompt.to_string())

Human: 
You are the official AI assistant for Govardhan Institute and Infotech.

Answer the user's question using ONLY the information in the PDF context.

Context:
[Page 2]
applications, REST APIs and AI-powered chat applications.
8. Certificates
For this demo knowledge base, assume that eligible students can receive a course or program completion
certificate after satisfying the applicable course requirements. Exact certificate rules should always be
confirmed with the administration.
9. Batch Information
Batch timing, course duration, fees, availability and start dates can vary by program and batch. The chatbot
should not invent these details when they are not present in the retrieved knowledge base.
10. Admission and Enrollment
Students interested in joining a program may ask about the course syllabus, duration, fees, schedule,
prerequisites, projects and certificate. If the knowledge base does not contain the requested information, the
assistant should direct the student to the ad

## 11. Send Context + Question to the LLM

In [13]:
response = llm.invoke(final_prompt)
print(response.content)

The knowledge base lists the following areas of study that Govardhan Institute & Infotech offers:

- **Programming fundamentals** – covering variables, data types, operators, control structures, functions, OOP, file handling, etc. (see Page 2, section 6)  
- **Web Development** – starting with HTML, CSS, JavaScript and progressing to Bootstrap, React, plus backend topics such as REST APIs, FastAPI, Django or Flask and database integration (see Page 2, section 5)  
- **Application Development** – building full‑stack applications, REST APIs and AI‑powered chat applications (mentioned on Page 2)  
- **Databases** – covered as part of the web‑development and application‑development tracks (Page 2)  
- **Data Analysis** – listed among the institute’s technology focus areas (Page 1)  
- **Artificial Intelligence** – including AI‑powered projects (Page 2)  
- **Generative AI** – referenced as a program focus (Page 2, testing questions)  
- **Related software technologies** – broader technolog

## 12. Build the complete RAG chain

In [14]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


## 13. Ask a question

In [15]:
question = "What courses are available at Govardhan Institute?"
answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: What courses are available at Govardhan Institute?

Answer: Based on the information in the demo knowledge base:

- **Programming Fundamentals** – covering variables, data types, operators, control structures, functions, arrays/lists, dictionaries, OOP, file handling, error handling, and problem‑solving. (Page 2, section 6)  
- **Web Development Program** – starting with HTML, CSS, and JavaScript and progressing to Bootstrap, React, and backend topics such as REST APIs, FastAPI, Django, Flask, plus database integration. (Page 2, section 5)  
- **Application Development** – mentioned as part of the institute’s overall training focus. (Page 1, bullet 1)  
- **Databases** – listed among the technology areas the institute trains in. (Page 1, bullet 1)  
- **Data Analysis** – listed among the technology areas the institute trains in. (Page 1, bullet 1)  
- **Artificial Intelligence** – listed among the technology areas the institute trains in. (Page 1, bullet 1)  
- **Generative A

## 14. Test insufficient information

In [20]:
question = "What is the weather in Surat today?"
answer = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:", answer)

Question: What is the weather in Surat today?

Answer: I don't have enough information in the provided institute information to answer this question. Please contact the Govardhan Institute and Infotech administration at 9898 57 68 77. (See Page 3 of the PDF.)


## 15. Interactive RAG chatbot

Type `exit` to stop.

In [ ]:
while True:
    question = input("\nAsk about Govardhan Institute & Infotech: ")

    if question.strip().lower() in ["exit", "quit", "bye"]:
        print("Goodbye!")
        break

    try:
        answer = rag_chain.invoke(question)
        print("\nAI:", answer)
    except Exception as e:
        print("\nError:", e)


Ask about Govardhan Institute & Infotech:  who you are ?



AI: I don't have enough information in the provided institute information to answer this question. Please contact the Govardhan Institute and Infotech administration at 9898 57 68 77.



Ask about Govardhan Institute & Infotech:  what is govardhan institute?



AI: Govardhan Institute & Infotech is a technology education and training institute (demo) located in Surat, Gujarat. It focuses on practical, career‑oriented technology education and hands‑on learning. 【Page 1】



Ask about Govardhan Institute & Infotech:  which courses are providing govardhan institute?


...


AI: The contact information for Govardhan Institute & Infotech is:

**Phone:** 9898 57 68 77  

(See Page 2 of the provided document.)



Ask about Govardhan Institute & Infotech:  can you give me the infomation of python coures?



AI: **Python Course Information (as per the provided knowledge base)**  

- **Program Area:** Programming (see Page 1, “Courses and Technologies”).  
- **Technologies/Topics Covered:** Python is listed alongside C, C++, and JavaScript as a core programming language taught in the institute’s programming courses.  

- **Programming Fundamentals (Page 6):** When Python is taught, the curriculum includes the typical fundamentals such as:  
  - Variables, data types, and operators  
  - Conditional statements and loops  
  - Functions, arrays/lists, and dictionaries  
  - Object‑oriented programming concepts  
  - File handling, error handling, and problem‑solving  

- **Generative AI Program (Page 1 & Page 4):** Python is also used in the demo Generative AI program, where students receive an introduction to Python fundamentals as needed, followed by topics like prompt engineering, large language models, LangChain, embeddings, vector stores, retrieval‑augmented generation (RAG), AI agents,


Ask about Govardhan Institute & Infotech:  what ia ducration of this coures and fees of this coures ?



AI: I don't have enough information in the provided institute information to answer this question. Please contact the Govardhan Institute and Infotech administration at **9898 57 68 77**. (See pages 2 and 11 of the PDF.)


## RAG Architecture

PDF
↓
PyPDFLoader
↓
Documents
↓
Text Chunks
↓
Embeddings
↓
FAISS Vector Store
↓
Retriever
↑
User Question
↓
Relevant Chunks
↓
Prompt (Context + Question)
↓
LLM
↓
Generated Answer

If information is insufficient:
→ Contact Administration
→ 9898 57 68 77